# 04 · Conversión de predicciones a COCO Keypoints para CVAT

**Objetivo:** convertir las predicciones de keypoints del modelo base (formato YOLO-pose) al formato COCO Keypoints, que CVAT puede importar, para revisar y corregir las anotaciones partiendo de las predicciones ya colocadas.

**Contexto:** revisar keypoints es más rápido si se parte de las predicciones del modelo en lugar de etiquetar desde cero. CVAT no importa YOLO-pose directamente, pero sí COCO Keypoints. Este notebook realiza esa conversión y prepara la carpeta de importación.

**Entrada:** predicciones en `data/keypoints_autolabel/` (formato YOLO-pose).

**Salida:** `data/cvat_import/annotations_coco.json` y carpeta de imágenes, listos para importar en CVAT.

## 1. Configuración y definición de keypoints

In [1]:
import json
import cv2
import shutil
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
AUTOLABEL_DIR = REPO_ROOT / "data" / "keypoints_autolabel"
IMAGES_DIR = AUTOLABEL_DIR / "images"
LABELS_DIR = AUTOLABEL_DIR / "labels"
OUTPUT_DIR = REPO_ROOT / "data" / "cvat_import"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Nombres de los 24 keypoints (mismo orden que el skeleton de CVAT y la conversión original)
KEYPOINT_NAMES = [
    "corner_inf_izq", "corner_sup_izq", "corner_inf_der", "corner_sup_der",
    "area_grande_izq_inf", "area_grande_izq_sup", "area_grande_der_inf", "area_grande_der_sup",
    "area_peq_izq_inf", "area_peq_izq_sup", "area_peq_der_inf", "area_peq_der_sup",
    "penalti_izq", "penalti_der", "centro", "media_inf", "media_sup",
    "fondo_izq_area_inf", "fondo_izq_area_sup", "fondo_der_area_inf",
    "poste_izq_inf", "poste_izq_sup", "poste_der_inf", "poste_der_sup",
]
N_KP = len(KEYPOINT_NAMES)
print(f"Keypoints definidos: {N_KP}")
print(f"Imágenes de entrada: {len(list(IMAGES_DIR.glob('*.jpg')))}")
print(f"Etiquetas de entrada: {len(list(LABELS_DIR.glob('*.txt')))}")

Keypoints definidos: 24
Imágenes de entrada: 120
Etiquetas de entrada: 120


## 2. Construcción de la estructura COCO

Se recorre cada par imagen-etiqueta y se construye el JSON COCO. Las coordenadas YOLO (normalizadas 0-1) se convierten a píxeles absolutos, que es lo que COCO espera. Cada keypoint se marca con visibilidad 2 (visible) o 0 (no visible), siguiendo el criterio del modelo.

In [2]:
coco = {
    "info": {"description": "Keypoints campo Bundesliga - predicciones modelo base"},
    "licenses": [],
    "images": [],
    "annotations": [],
    "categories": [{
        "id": 1,
        "name": "field",
        "supercategory": "field",
        "keypoints": KEYPOINT_NAMES,
        "skeleton": []
    }]
}

ann_id = 1
img_id = 1
label_files = sorted(LABELS_DIR.glob("*.txt"))

for label_path in label_files:
    img_path = IMAGES_DIR / (label_path.stem + ".jpg")
    if not img_path.exists():
        continue

    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    coco["images"].append({
        "id": img_id,
        "file_name": img_path.name,
        "width": w,
        "height": h
    })

    with open(label_path) as f:
        valores = f.read().split()

    kp_vals = valores[5:]  # saltar clase + bbox
    keypoints_coco = []
    num_visibles = 0
    for i in range(N_KP):
        x_n, y_n, v = float(kp_vals[i*3]), float(kp_vals[i*3+1]), int(kp_vals[i*3+2])
        if v == 2:
            keypoints_coco.extend([x_n * w, y_n * h, 2])
            num_visibles += 1
        else:
            keypoints_coco.extend([0, 0, 0])

    coco["annotations"].append({
        "id": ann_id,
        "image_id": img_id,
        "category_id": 1,
        "keypoints": keypoints_coco,
        "num_keypoints": num_visibles,
        "bbox": [0, 0, w, h],
        "area": w * h,
        "iscrowd": 0
    })
    ann_id += 1
    img_id += 1

print(f"Imágenes procesadas: {len(coco['images'])}")
print(f"Anotaciones generadas: {len(coco['annotations'])}")

Imágenes procesadas: 120
Anotaciones generadas: 120


## 3. Guardado del JSON y copia de imágenes

In [3]:
# Guardar el JSON COCO
coco_path = OUTPUT_DIR / "annotations_coco.json"
with open(coco_path, "w", encoding="utf-8") as f:
    json.dump(coco, f, indent=2)

# Copiar las imágenes a la carpeta de importación
cvat_images = OUTPUT_DIR / "images"
cvat_images.mkdir(exist_ok=True)
for img_path in IMAGES_DIR.glob("*.jpg"):
    shutil.copy2(img_path, cvat_images / img_path.name)

print(f"JSON COCO guardado en: {coco_path}")
print(f"Imágenes copiadas a: {cvat_images}")
print(f"Total imágenes copiadas: {len(list(cvat_images.glob('*.jpg')))}")
print("\nListo para importar en CVAT")

JSON COCO guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\cvat_import\annotations_coco.json
Imágenes copiadas a: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\cvat_import\images
Total imágenes copiadas: 120

Listo para importar en CVAT


## 4. Conclusiones

Tras este notebook:

- Las predicciones del modelo base se han convertido al formato COCO Keypoints.
- La carpeta `data/cvat_import/` contiene el JSON de anotaciones y las imágenes.
- El siguiente paso es subir las imágenes a una tarea de CVAT (dentro del proyecto con el skeleton de 24 puntos ya definido) e importar el JSON COCO para revisar las anotaciones partiendo de las predicciones.